In [ ]:
import pandas as pd
import re

# Define merge keys
keys = ["REF_AREA", "REF_AREA_LABEL"]
drop_cols = []  # Add columns to drop if needed

pm25 = pd.read_csv("..//data//pm25.csv")
gdp = pd.read_csv("..//data//gdp.csv")
population = pd.read_csv("..//data//population.csv")

# Extract years from pm25 columns
pm25_years = set()
for c in pm25.columns:
    m = re.search(r"(\d{4})", c)
    if m:
        pm25_years.add(m.group(1))

print(f"Years in PM2.5 data: {sorted(pm25_years)}")

def prefix_year_columns(df, prefix, keep_keys=keys, filter_years=None):
    df = df.copy()
    rename = {}
    year_new_names = []
    for c in df.columns:
        m = re.search(r"(\d{4})", c)
        if m:
            year = m.group(1)
            # Only include if filter_years is None or year is in filter_years
            if filter_years is None or year in filter_years:
                rename[c] = f"{prefix}_{year}"
                year_new_names.append(rename[c])
    df = df.rename(columns=rename)
    cols = [k for k in keep_keys if k in df.columns] + [c for c in year_new_names if c in df.columns]
    return df[cols]

pm25_pref = prefix_year_columns(pm25, "pm25")
gdp_pref = prefix_year_columns(gdp, "gdp", filter_years=pm25_years)
pop_pref = prefix_year_columns(population, "pop", filter_years=pm25_years)

pm25_enriched = pm25_pref.merge(gdp_pref, on=keys, how="left").merge(pop_pref, on=keys, how="left")

to_drop = [c for c in drop_cols if c in pm25_enriched.columns]
if to_drop:
    pm25_enriched = pm25_enriched.drop(columns=to_drop)

print(pm25_enriched.shape)
pm25_enriched.head()

pm25_enriched.to_csv("../data/processed_data.csv", index=False)

NameError: name 'keys' is not defined